In [13]:
import warnings
warnings.filterwarnings('ignore')

import os
import geopandas as gpd
import snman
import pandas as pd

PROJECT = '_main'

data_directory = os.path.join('C:',os.sep,'Users','shiry', 'snman_sgProject')
inputs_path = os.path.join(data_directory,'custom_inputs')
process_path = os.path.join(data_directory, 'process', PROJECT)
export_path = os.path.join(data_directory, 'outputs', PROJECT)

## Load Data

In [2]:
rebuilt_network = gpd.read_file(os.path.join(export_path, 'lane_geometries_rebuilt_edges.gpkg'))
og_network = gpd.read_file(os.path.join(process_path, 'lane_geometries_edges.gpkg'))

In [3]:
lta_cycling_network = gpd.read_file(
    os.path.join(inputs_path, 'geospatial_LTA', 'CyclingPath_Aug2025', 'CyclingPathGazette.shp')
)

## Comparison of Bike Infrastructure
Visual comparison of off-street infrastructure vs rebuilt network for on-street cycling

In [4]:
lta_cycling_network = lta_cycling_network[lta_cycling_network['PLANNING_1'] == 'JURONG WEST']
lta_cycling_network.explore()

In [5]:
rebuilt_network_cycling = rebuilt_network[rebuilt_network['lanetype'] == 'L']
rebuilt_network_cycling.explore()

## Network Indicators

In [14]:
L = snman.io.load_street_graph(
    os.path.join(export_path, 'G_edges.gpkg'),
    os.path.join(export_path, 'G_nodes.gpkg'),
    crs=3414
)

stats = snman.stats.network_metrics(L)
stats

label,CARS BEFORE,CARS AFTER,CYCLING BEFORE,CYCLING AFTER,TRANSIT BEFORE,TRANSIT AFTER
_L,"(251, 0, 1132, 9, 1, 32, 157, 34, 2, 158, 244,...","(251, 0, 1132, 9, 1, 32, 157, 34, 2, 158, 244,...","(269, 16, 310, 536, 537, 538, 539, 17, 254, 25...","(269, 16, 310, 536, 537, 538, 539, 254, 17, 25...","(251, 0, 1132, 9, 1, 32, 157, 34, 2, 158, 244,...","(251, 0, 1132, 9, 1, 32, 157, 34, 2, 158, 244,..."
_mode,private_cars,private_cars,cycling,cycling,transit,transit
usable_N_nodes,1162,1173,1118,1145,1162,1173
usable_N_edges,2965,2032,2766,3123,2965,2032
convex_hull_m2,17001185.527998,17095548.939315,15826512.372802,16469949.446849,17001185.527998,17095548.939315
usable_lane_m,426229.391343,298341.095156,399022.333945,447437.752975,426229.391343,298341.095156
usable_lane_surface_m2,1219175.063616,826713.495936,1108612.108253,1147514.23013,1219175.063616,826713.495936
as_primary_mode_lane_m,426229.391343,298341.095156,16618.216229,183739.666819,0.0,0.0
as_primary_mode_lane_surface_m2,1219175.063616,826713.495936,20772.770286,424729.761193,0.0,0.0
avg_betweenness_centrality_norm,0.007294,0.011282,0.006849,0.005598,0.007294,0.011282


In [43]:
network_indicators = stats[5:12].copy()
network_indicators = network_indicators.reset_index().rename(columns={'index':'metric'})

network_indicators = pd.melt(network_indicators, id_vars='metric', value_vars=['CARS BEFORE', 'CARS AFTER', 'CYCLING BEFORE', 'CYCLING AFTER'])

# change into more readable format
network_indicators[['mode', 'scenario']] = network_indicators['label'].str.split(' ', n=1,  expand=True)
network_indicators = pd.pivot_table(network_indicators, index=['metric', 'mode'], columns='scenario', values='value').reset_index()
network_indicators = network_indicators.rename(columns={'BEFORE':'Status Quo', 'AFTER':'Rebuilt'})

# add the change percentage column
network_indicators['Change'] = ((network_indicators['Rebuilt'] - network_indicators['Status Quo'])
                                / network_indicators['Status Quo'] *100)

# add mode labels to metric column
network_indicators['metric'] = network_indicators['metric'] + ' (' + network_indicators['mode'].str.lower() + ')'
network_indicators = network_indicators[['metric', 'Status Quo', 'Rebuilt', 'Change']]

# units
is_meters = network_indicators['metric'].str.contains('_m ', '_m2 ')
network_indicators.loc[is_meters, ['Status Quo', 'Rebuilt']] /= 1000
network_indicators['metric'] = network_indicators['metric'].str.replace('_m2 ', '_km2 ', regex=False)
network_indicators['metric'] = network_indicators['metric'].str.replace('_m ', '_km ', regex=False)

network_indicators

scenario,metric,Status Quo,Rebuilt,Change
0,as_primary_mode_lane_km (cars),426.229391,298.341095,-30.00457
1,as_primary_mode_lane_km (cycling),16.618216,183.739667,1005.652161
2,as_primary_mode_lane_surface_km2 (cars),1219175.063616,826713.495936,-32.190748
3,as_primary_mode_lane_surface_km2 (cycling),20772.770286,424729.761193,1944.646695
4,avg_betweenness_centrality_norm (cars),0.007294,0.011282,54.684954
5,avg_betweenness_centrality_norm (cycling),0.006849,0.005598,-18.270822
6,avg_shortest_path_km (cars),3.15425,3.747342,18.802934
7,avg_shortest_path_km (cycling),2.968804,2.800993,-5.652454
8,avg_shortest_path_vod_km (cars),3.15425,3.747342,18.802934
9,avg_shortest_path_vod_km (cycling),3.357443,1.969276,-41.345972
